In [11]:
%load_ext autoreload
%autoreload 2

import sys
import os
import math
from importlib import reload

import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import mars
from mars import spin_model, spectra_manager, constants, population, concat
from mars.population import RedfieldRelaxationChannel, RedfieldManager
# Physical copulatiu for unit conversions
H_PLANCK =LTZMANN = 1.380649e-23  # J/K
HZ_TO_RAD_S = 2 * math.pi

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import torch
import numpy as np
import math
from typing import Optional, List, Tuple
from mars.population.contexts import Context, SummedContext
from mars.population import transform
from mars import spin_model
import mars

# 1. Create Samples for Checking

In [13]:
def create_samples():
    g_tensor_1 = spin_model.Interaction((2.02, 2.04, 2.06), dtype=dtype)
    zfs_1 = spin_model.DEInteraction((200 * 1e6, 50 * 1e6), dtype=dtype)  # D=200 MHz, E=50 MHz
    g_tensor_2 = spin_model.Interaction((2.02, 2.04, 2.06), dtype=dtype)
    zfs_2 = spin_model.DEInteraction((200 * 1e6, 50 * 1e6), dtype=dtype)
    
    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[g_tensor_1],
        electron_electron=[(0, 0, zfs_1)],
        dtype=dtype,
    )
    sample_1 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
    )


    g_tensor = spin_model.Interaction((2.02, 2.14, 2.16), dtype=dtype)
    zfs = spin_model.DEInteraction((200 * 1e6, 70 * 1e6), dtype=dtype)
    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[g_tensor_2],
        electron_electron=[(0, 0, zfs_2)],
        dtype=dtype
    )
    sample_2 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
)
    return sample_1, sample_2

def get_eigen_basis(sample, filed: float):
    magnetic_field = torch.tensor(filed)
    F, _, _, Gz = sample.get_hamiltonian_terms()
    H = F + Gz * magnetic_field
    H = H.unsqueeze(-3)
    values, vectors = torch.linalg.eigh(H)
    return vectors

In [14]:
def _hz_to_energy_joules(energies_hz: torch.Tensor) -> torch.Tensor:
    """Convert energies from Hz to Joules."""
    return energies_hz * H_PLANCK

def _compute_boltzmann_distribution(
    energies_hz: torch.Tensor, 
    temperature: torch.Tensor
) -> torch.Tensor:
    """
    Compute Boltzmann distribution from energies in Hz.
    Returns normalized populations.
    """
    exp_factor = torch.exp(-mars.constants.unit_converter(energies_hz, "Hz_to_K") / temperature)
    return exp_factor / exp_factor.sum(dim=-1, keepdim=True)


def _create_redfield_channel(
    dim: int,
    spectral_density_func: callable,
    thermal_mode: str = "symmetric",
    secular: bool = True
) -> 'RedfieldRelaxationChannel':
    """
    Create a concrete RedfieldRelaxationChannel for testing.
    Since RedfieldRelaxationChannel is abstract, this creates a minimal implementation.
    """
    # Create simple coupling operators (e.g., Sx-like)
    ops = []
    # Operator 1: Simple lowering-like operator
    op1 = torch.zeros(dim, dim)
    for i in range(dim - 1):
        op1[i, i+1] = 1.0
        op1[i+1, i] = 1.0
    ops.append((op1, None))  # (static, field-dependent)
    channel = RedfieldRelaxationChannel(ops, spectral_density_func, thermal_mode)
    #channel.post_init(eigen_basis_flag=False, secular=secular)
    return channel


def _create_redfield_manager(
    dim: int,
    spectral_density_func: callable,
    thermal_mode: str = "symmetric",
    secular: bool = True
) -> 'RedfieldManager':
    """Create a RedfieldManager with a single test channel."""
    channel = _create_redfield_channel(dim, spectral_density_func, thermal_mode, secular)
    return RedfieldManager([channel])


def _create_context_with_redfield(
    sample,
    basis,
    redfield_manager: 'RedfieldManager',
    init_populations: Optional[List[float]] = None,
    dtype: torch.dtype = torch.float64
) -> Context:
    """
    Create a Context with Redfield relaxation manager attached.
    Adjust this to match your actual Context API for attaching relaxation.
    """
    context = Context(
        basis=basis,
        sample=sample,
        init_populations=init_populations if init_populations else [1.0/3.0] * 3,
        dtype=dtype
    )
    # Attach Redfield manager - adjust based on your Context API
    # This might be: context.relaxation_manager = redfield_manager
    # Or passed in __init__: context = Context(..., relaxation_manager=redfield_manager)
    context._relaxation_manager = redfield_manager
    return context

# ============================================================================
# TEST 1: TRACE CONSERVATION
# ============================================================================

def check_redfield_trace_conservation(
    context: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-5
) -> None:
    """
    Validate that the Redfield superoperator preserves the trace of the density matrix.
    
    Physical Rule:
    Tr(dρ/dt) = Tr(L ρ) = 0 for any valid relaxation superoperator L.
    In vectorized form (row-major), the sum of rows corresponding to diagonal 
    elements (populations) must be zero.
    """
    manager = context._relaxation_manager
    superop = manager.compute_relaxation_superoperator(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    dim = context.spin_system_dim
    dim_sq = dim * dim
    
    pop_indices = torch.arange(dim, device=superop.device) * dim + torch.arange(dim, device=superop.device)
    row_sum = superop[..., pop_indices, :].sum(dim=-2)
    
    assert torch.allclose(row_sum, torch.zeros_like(row_sum), atol=atol), \
        f"Redfield superoperator does not conserve trace. Max deviation: {row_sum.abs().max().item():.2e}"
    
    print("✓ Redfield trace conservation test passed")


# ============================================================================
# TEST 2: DETAILED BALANCE
# ============================================================================

def check_redfield_detailed_balance(
    context: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-4
) -> None:
    """
    Validate that the transition rates satisfy detailed balance at thermal equilibrium.
    
    Physical Rule:
    W_ac * P_c_eq = W_ca * P_a_eq
    => W_ac / W_ca = exp(-(E_a - E_c) / k_B T)
    """
    if not hasattr(context, '_relaxation_manager') or context._relaxation_manager is None:
        print("⊘ Redfield detailed balance test skipped: no Redfield manager attached")
        return
    
    manager = context._relaxation_manager
    rates = manager.compute_transition_probabilities(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    dim = context.spin_system_dim
    
    # Calculate Boltzmann distribution
    boltzmann_pop = _compute_boltzmann_distribution(energies, temperature)

    # Check detailed balance: W_ac * P_c_eq == W_ca * P_a_eq
    # W[a, c] is rate c -> a
    # Detailed balance: W[a,c] * pop[c] == W[c,a] * pop[a]
    
    lhs = rates * boltzmann_pop.unsqueeze(-2)  # W_ac * P_c
    rhs = rates.transpose(-1, -2) * boltzmann_pop.unsqueeze(-1)  # W_ca * P_a
    
    diff = (lhs - rhs).abs()
    
    # Only check where rates are significant (avoid division by zero issues)
    mask = rates > 1e-10
    diff_masked = diff * mask
    
    assert torch.all(diff_masked < atol), \
        f"Detailed balance violated. Max deviation: {diff_masked.max().item():.2e}"
    
    print("✓ Redfield detailed balance test passed")


# ============================================================================
# TEST 3: REDFIELD VS LINDBLAD POPULATION RATES
# ============================================================================

def check_redfield_vs_lindblad_rates(
    context: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-5
) -> None:
    """
    Compare Redfield population rates with equivalent Lindblad kinetic coefficients.
    
    In the secular approximation, Redfield population dynamics are equivalent to 
    a Lindblad master equation with jump operators L_ac = |a><c| and rates gamma_ac = W_ac.
    """
    if not hasattr(context, '_relaxation_manager') or context._relaxation_manager is None:
        print("⊘ Redfield vs Lindblad rates test skipped: no Redfield manager attached")
        return
    
    manager = context._relaxation_manager
    redfield_rates = manager.compute_transition_probabilities(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    dim = context.spin_system_dim
    
    # Construct Lindblad population rate matrix from Redfield rates
    # d/dt rho_aa = sum_c ( W_ac * rho_cc - W_ca * rho_aa )
    lindblad_rates = torch.zeros_like(redfield_rates)
    
    # Off-diagonal: Gain from c to a (W_ac)
    # Redfield already has W[a,c] = rate c->a with zero diagonal
    lindblad_rates[..., :, :] = redfield_rates
    
    # Diagonal: Loss from a (sum of rates OUT of state a)
    # Loss term for state a is sum_k W[k, a] (all transitions from a to k)
    loss_rates = redfield_rates.sum(dim=-2, keepdim=True)  # Sum over destination k
    for i in range(dim):
        lindblad_rates[..., i, i] = -loss_rates[..., 0, i]
    
    # Redfield rates should have zero diagonal already
    # Check that our construction matches
    assert torch.allclose(redfield_rates, lindblad_rates, atol=atol), \
        "Redfield rates do not match constructed Lindblad kinetic coefficients"
    
    print("✓ Redfield vs Lindblad rates consistency test passed")


# ============================================================================
# TEST 4: REDFIELD VS LINDBLAD SUPEROPERATOR
# ============================================================================

def check_redfield_vs_lindblad_superoperator(
    context: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-4
) -> None:
    """
    Compare the full Redfield Superoperator with a manually constructed Lindblad Superoperator.
    
    This validates that the coherence decay terms (dephasing) are also consistent 
    with the Lindblad form derived from the transition rates.
    """
    if not hasattr(context, '_relaxation_manager') or context._relaxation_manager is None:
        print("⊘ Redfield vs Lindblad superoperator test skipped: no Redfield manager attached")
        return
    
    manager = context._relaxation_manager
    dim = context.spin_system_dim
    dim_sq = dim * dim
    
    # 1. Get Redfield Superoperator
    R_redfield = manager.compute_relaxation_superoperator(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    # 2. Get population rates
    W = manager.compute_transition_probabilities(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    # 3. Construct Lindblad Superoperator from rates
    R_lindblad = torch.zeros_like(R_redfield)
    
    # Population block (diagonal elements of density matrix)
    pop_indices = torch.arange(dim, device=R_redfield.device)
    for a in range(dim):
        for c in range(dim):
            if a != c:
                # Gain term: rho_cc -> rho_aa with rate W_ac
                idx_in = c * dim + c
                idx_out = a * dim + a
                R_lindblad[..., idx_out, idx_in] = W[..., a, c]
        
        # Loss term: rho_aa -> rho_aa with rate -sum_k W_ka
        idx_in = a * dim + a
        idx_out = a * dim + a
        R_lindblad[..., idx_out, idx_in] = -W[..., :, a].sum(dim=-1)
    
    # 4. Compare population blocks
    pop_idx_flat = pop_indices * dim + pop_indices
    R_pop_red = R_redfield[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
    R_pop_lin = R_lindblad[..., pop_idx_flat.unsqueeze(-1), pop_idx_flat.unsqueeze(-2)]
    
    assert torch.allclose(R_pop_red, R_pop_lin, atol=atol), \
        "Population block of Redfield superoperator does not match Lindblad construction"
    
    # 5. Check coherence decay diagonal (for secular case)
    # Coherence rho_ab corresponds to index a*dim + b (a != b)
    coh_indices = []
    for a in range(dim):
        for b in range(dim):
            if a != b:
                coh_indices.append(a * dim + b)
    
    if coh_indices:
        coh_idx_tensor = torch.tensor(coh_indices, dtype=torch.long, device=R_redfield.device)
        # Check diagonal elements for coherences
        R_coh_red_diag = R_redfield[..., coh_idx_tensor, coh_idx_tensor]
        # Should be negative (decay rates)
        assert torch.all(R_coh_red_diag.real <= 1e-10), \
            f"Coherence decay rates should be negative. Max: {R_coh_red_diag.real.max().item():.2e}"
    
    print("✓ Redfield vs Lindblad superoperator consistency test passed")


# ============================================================================
# TEST 5: SECULAR VS NON-SECULAR STRUCTURE
# ============================================================================

def check_redfield_secular_structure(
    context_secular: Context,
    context_non_secular: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-6
) -> None:
    """
    Validate the structural differences between Secular and Non-Secular Redfield tensors.
    
    Physical Rule:
    Secular approximation zeros out terms where omega_ab != omega_cd.
    Specifically, it decouples populations from coherences and different coherences from each other.
    """
    if not hasattr(context_secular, '_relaxation_manager') or context_secular._relaxation_manager is None:
        print("⊘ Secular structure test skipped: no Redfield manager attached")
        return
    
    manager_sec = context_secular._relaxation_manager
    manager_non_sec = context_non_secular._relaxation_manager
    
    R_sec = manager_sec.compute_relaxation_superoperator(
        transformation_unitary=context_secular.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    R_non_sec = manager_non_sec.compute_relaxation_superoperator(
        transformation_unitary=context_non_secular.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    dim = context_secular.spin_system_dim
    dim_sq = dim * dim
    
    # 1. Check Secular Sparsity
    # In Secular, R_ab,cd should be 0 unless (a=b and c=d) OR (a=c and b=d)
    mask_secular_allowed = torch.zeros(dim_sq, dim_sq, dtype=torch.bool, device=R_sec.device)
    for a in range(dim):
        for b in range(dim):
            for c in range(dim):
                for d in range(dim):
                    idx_out = a * dim + b
                    idx_in = c * dim + d
                    if (a == b and c == d) or (a == c and b == d):
                        mask_secular_allowed[idx_out, idx_in] = True
    
    # Check if Secular Superoperator has non-zero elements outside allowed mask
    sec_non_zero = R_sec.abs() > atol
    violation = sec_non_zero & (~mask_secular_allowed)
    
    assert not violation.any(), \
        f"Secular superoperator has non-zero elements outside secular approximation mask. Count: {violation.sum().item()}"
    
    # 2. Check Non-Secular has MORE non-zero elements (usually)
    nnz_sec = (R_sec.abs() > atol).sum().item()
    nnz_non_sec = (R_non_sec.abs() > atol).sum().item()
    
    print(f"  Secular non-zero elements: {nnz_sec}, Non-Secular: {nnz_non_sec}")
    assert nnz_non_sec >= nnz_sec, "Non-secular should have at least as many terms as secular"
    
    print("✓ Redfield secular structure test passed")


# ============================================================================
# TEST 6: TEMPERATURE DEPENDENCE
# ============================================================================

def check_redfield_temperature_dependence(
    context: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    atol: float = 1e-5
) -> None:
    """
    Validate that relaxation rates change appropriately with temperature.
    
    Physical Rule:
    Higher temperature generally increases upward transition rates (absorption)
    due to increased bath occupation numbers.
    """
    if not hasattr(context, '_relaxation_manager') or context._relaxation_manager is None:
        print("⊘ Temperature dependence test skipped: no Redfield manager attached")
        return
    
    manager = context._relaxation_manager
    
    T_low = torch.tensor(1.0, device=energies.device, dtype=energies.dtype)
    T_high = torch.tensor(300.0, device=energies.device, dtype=energies.dtype)
    
    rates_low = manager.compute_transition_probabilities(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=T_low
    )
    
    rates_high = manager.compute_transition_probabilities(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=T_high
    )
    
    # 1. Rates should not be identical
    assert not torch.allclose(rates_low, rates_high, atol=atol), \
        "Rates should change with temperature"
    
    # 2. Upward transitions (omega < 0) should increase more with T
    # omega[i,j] = E_j - E_i, so omega < 0 means E_j < E_i (upward transition j->i)
    omega_hz = energies.unsqueeze(-1) - energies.unsqueeze(-2)
    upward_mask = omega_hz < 0
    
    upward_low = rates_low[upward_mask].mean()
    upward_high = rates_high[upward_mask].mean()
    
    # Upward rates should increase with temperature
    assert upward_high > upward_low * 0.9, \
        f"Upward rates should increase with temperature: {upward_low:.2e} -> {upward_high:.2e}"
    
    print(f"  Upward rates: T=1K: {upward_low:.2e}, T=300K: {upward_high:.2e}")
    print("✓ Redfield temperature dependence test passed")


# ============================================================================
# TEST 7: STEADY STATE CONVERGENCE
# ============================================================================

def check_redfield_steady_state(
    context: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-4
) -> None:
    """
    Validate that the Redfield dynamics converge to the thermal equilibrium state.
    
    Physical Rule:
    At steady state (t -> ∞), the density matrix should approach the Boltzmann distribution.
    L @ rho_ss = 0
    """
    if not hasattr(context, '_relaxation_manager') or context._relaxation_manager is None:
        print("⊘ Steady state test skipped: no Redfield manager attached")
        return
    
    manager = context._relaxation_manager
    dim = context.spin_system_dim
    
    # Get superoperator
    superop = manager.compute_relaxation_superoperator(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    # Compute expected thermal equilibrium state (Boltzmann distribution)
    boltzmann_pop = _compute_boltzmann_distribution(energies, temperature)
    
    # Vectorize the thermal state (diagonal density matrix)
    rho_thermal = torch.zeros(dim * dim, device=superop.device, dtype=superop.dtype)
    for i in range(dim):
        rho_thermal[i * dim + i] = boltzmann_pop[i]
    
    # Apply superoperator: should give zero (steady state)
    d_rho_dt = superop @ rho_thermal.unsqueeze(-1)
    
    # Check that derivative is approximately zero
    assert torch.allclose(d_rho_dt, torch.zeros_like(d_rho_dt), atol=atol), \
        f"Thermal state is not a steady state. Max |dρ/dt|: {d_rho_dt.abs().max().item():.2e}"
    
    print("✓ Redfield steady state test passed")


# ============================================================================
# TEST 8: INTEGRATION WITH CONTEXT OPERATIONS
# ============================================================================

def check_redfield_context_integration(
    context: Context,
    full_system_vectors: torch.Tensor,
    fields: torch.Tensor,
    energies: torch.Tensor,
    temperature: torch.Tensor,
    atol: float = 1e-5
) -> None:
    """
    Validate that Redfield relaxation integrates correctly with Context operations.
    
    Tests:
    - get_transformed_free_superop returns Redfield superoperator
    - get_transformed_free_probs returns Redfield transition rates
    - Operations like @ (multiplication) and + (addition) work with Redfield contexts
    """
    if not hasattr(context, '_relaxation_manager') or context._relaxation_manager is None:
        print("⊘ Context integration test skipped: no Redfield manager attached")
        return
    
    # 1. Check superoperator from context matches Redfield manager
    context_superop = context.get_transformed_free_superop(full_system_vectors)
    
    manager_superop = context._relaxation_manager.compute_relaxation_superoperator(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    if context_superop is not None:
        assert torch.allclose(context_superop, manager_superop, atol=atol), \
            "Context superoperator does not match Redfield manager output"
    
    # 2. Check rate matrix from context matches Redfield manager
    context_rates = context.get_transformed_free_probs(full_system_vectors)
    
    manager_rates = context._relaxation_manager.compute_transition_probabilities(
        transformation_unitary=context.basis,
        fields=fields,
        energies=energies,
        temperature=temperature
    )
    
    if context_rates is not None:
        assert torch.allclose(context_rates, manager_rates, atol=atol), \
            "Context rate matrix does not match Redfield manager output"
    
    print("✓ Redfield context integration test passed")


In [ ]:
# ============================================================================
# MAIN TEST RUNNER
# ============================================================================

def run_all_redfield_tests(
    sample_1,
    sample_2,
    basis_1,
    basis_2,
    field_strength: float = 0.34,
    temperature: float = 300.0,
    dtype: torch.dtype = torch.float64
) -> None:
    """
    Run all Redfield relaxation tests with the provided samples and bases.
    """
    print("\n" + "="*70)
    print("REDFIELD RELAXATION TEST SUITE")
    print("="*70 + "\n")
    
    # Create contexts with Redfield managers
    dim = 3  # Based on your sample creation (3-level system)
    
    # Simple spectral density for testing (constant)
    def spectral_density(omega_rad_s: torch.Tensor) -> torch.Tensor:
        return torch.ones_like(omega_rad_s) * 1e6  # rad/s
    
    # Create Redfield managers
    manager_secular = _create_redfield_manager(dim, spectral_density, "symmetric", secular=True)
    manager_non_secular = _create_redfield_manager(dim, spectral_density, "symmetric", secular=False)
    
    # Create contexts
    context_sec = _create_context_with_redfield(sample_1, basis_1, manager_secular, dtype=dtype)
    context_non_sec = _create_context_with_redfield(sample_1, basis_1, manager_non_secular, dtype=dtype)
    
    # Get Hamiltonian terms for energies and fields
    F, _, _, Gz = sample_1.get_hamiltonian_terms()
    magnetic_field = torch.tensor(field_strength, dtype=dtype)
    H = F + Gz * magnetic_field
    H = H.unsqueeze(-3)
    energies, vectors = torch.linalg.eigh(H)
    energies = energies.squeeze(-1)  # Shape: (..., N)
    
    # Full system vectors (eigenbasis)
    full_system_vectors = vectors
    
    # Fields tensor (for field-dependent operators)
    fields = magnetic_field.unsqueeze(-1).unsqueeze(-1)  # Shape: (1, 1, 1)
    
    # Temperature
    temperature_tensor = torch.tensor(temperature, dtype=dtype)
    
    print(f"System dimension: {dim}")
    print(f"Magnetic field: {field_strength} T")
    print(f"Temperature: {temperature} K")
    print(f"Energy levels (Hz): {energies.squeeze()}\n")
    
    # Run tests
    print("-"*70)
    print("TEST 1: Trace Conservation")
    print("-"*70)
    check_redfield_trace_conservation(
        context_sec, full_system_vectors, fields, energies, temperature_tensor
    )
    
    print("\n" + "-"*70)
    print("TEST 2: Detailed Balance")
    print("-"*70)
    check_redfield_detailed_balance(
        context_sec, full_system_vectors, fields, energies, temperature_tensor
    )
    
    print("\n" + "-"*70)
    print("TEST 3: Redfield vs Lindblad Rates")
    print("-"*70)
    check_redfield_vs_lindblad_rates(
        context_sec, full_system_vectors, fields, energies, temperature_tensor
    )
    
    print("\n" + "-"*70)
    print("TEST 4: Redfield vs Lindblad Superoperator")
    print("-"*70)
    check_redfield_vs_lindblad_superoperator(
        context_sec, full_system_vectors, fields, energies, temperature_tensor
    )
    
    print("\n" + "-"*70)
    print("TEST 5: Secular vs Non-Secular Structure")
    print("-"*70)
    check_redfield_secular_structure(
        context_sec, context_non_sec, full_system_vectors, 
        fields, energies, temperature_tensor
    )
    
    print("\n" + "-"*70)
    print("TEST 6: Temperature Dependence")
    print("-"*70)
    check_redfield_temperature_dependence(
        context_sec, full_system_vectors, fields, energies
    )
    
    print("\n" + "-"*70)
    print("TEST 7: Steady State Convergence")
    print("-"*70)
    check_redfield_steady_state(
        context_sec, full_system_vectors, fields, energies, temperature_tensor
    )
    
    print("\n" + "-"*70)
    print("TEST 8: Context Integration")
    print("-"*70)
    check_redfield_context_integration(
        context_sec, full_system_vectors, fields, energies, temperature_tensor
    )
    
    print("\n" + "="*70)
    print("ALL REDFIELD TESTS COMPLETED")
    print("="*70 + "\n")


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    dtype = torch.float64
    
    # Create samples (from your provided code)
    sample_1, sample_2 = create_samples()
    
    # Get eigenbases
    basis_1 = get_eigen_basis(sample_1, filed=0.34)
    basis_2 = get_eigen_basis(sample_2, filed=0.34)
    
    # Run all tests
    run_all_redfield_tests(
        sample_1=sample_1,
        sample_2=sample_2,
        basis_1=basis_1,
        basis_2=basis_2,
        field_strength=0.34,
        temperature=300.0,
        dtype=dtype
    )